# PlantVillage — Plant Disease Classification Using 12 Machine Learning Algorithms

**Dataset:** PlantVillage `raw/color/` RGB images  
**Approach:** HOG + colour histogram feature extraction, followed by classical ML classification.

> Important: This notebook expects one subfolder per class, for example `raw/color/Tomato___Early_blight/`. If your downloaded folder has a different structure, update `DATASET_DIR` in Block 2.  
> For a quick trial, set `MAX_IMAGES_PER_CLASS = 100`. Set it to `None` to use every image. Full-dataset feature extraction can take significant time and memory.
>
> **Data leakage note:** PlantVillage includes leaf-grouping metadata because images of the same leaf may be related. A random image-level split can overestimate performance if related images land in different splits. This notebook uses stratified image-level splits as a baseline. For a research-grade result, use the repository's `leaf_grouping` metadata to keep each leaf group entirely within one split, if that metadata is available and mapped to the image filenames.

## Block 1 — Install and import required libraries

In [ ]:
# Run this cell if the required libraries are not installed
# %pip install numpy pandas matplotlib seaborn scikit-learn scikit-image pillow joblib tqdm

import os
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
from tqdm.auto import tqdm
from skimage.feature import hog
from skimage.color import rgb2gray

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier,
    AdaBoostClassifier, BaggingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.base import clone
import joblib

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
RANDOM_STATE = 42

print("Libraries imported successfully.")

## Block 2 — Set the dataset folder

In [ ]:
# CHANGE THIS PATH to the location of your extracted PlantVillage raw/color folder.
# Windows example:
DATASET_DIR = Path(r"D:\PlantVillage-Dataset\raw\color")

# If you downloaded only the color folder, point DATASET_DIR directly to that folder.
# Example: Path(r"C:\Users\YourName\Downloads\color")

# Set to 100 for a quick test run; set to None to use all images in each class.
MAX_IMAGES_PER_CLASS = 100

IMAGE_SIZE = (128, 128)
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATASET_DIR}\n"
        "Update DATASET_DIR in Block 2 with the correct path."
    )

print("Dataset directory:", DATASET_DIR)

## Block 3 — Discover image classes and files

In [ ]:
class_folders = sorted([p for p in DATASET_DIR.iterdir() if p.is_dir()])
if not class_folders:
    raise ValueError("No class subfolders found. Expected one folder per disease class.")

records = []
for class_folder in class_folders:
    image_paths = [
        p for p in class_folder.rglob("*")
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    image_paths = sorted(image_paths)
    if MAX_IMAGES_PER_CLASS is not None:
        image_paths = image_paths[:MAX_IMAGES_PER_CLASS]
    for image_path in image_paths:
        records.append({"image_path": str(image_path), "label": class_folder.name})

image_df = pd.DataFrame(records)
if image_df.empty:
    raise ValueError("No images found. Check folder structure and supported file extensions.")

print("Total images selected:", len(image_df))
print("Number of classes:", image_df["label"].nunique())
display(image_df.head())
display(image_df["label"].value_counts().rename("image_count").to_frame())

## Block 4 — Visualise class distribution

In [ ]:
class_counts = image_df["label"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(14, max(6, len(class_counts) * 0.35)))
sns.barplot(x=class_counts.values, y=class_counts.index)
plt.title("PlantVillage Image Count per Class")
plt.xlabel("Number of images")
plt.ylabel("Disease class")
plt.tight_layout()
plt.show()

## Block 5 — Display sample images

In [ ]:
sample_classes = image_df["label"].drop_duplicates().head(6).tolist()
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, class_name in zip(axes.ravel(), sample_classes):
    sample_path = image_df.loc[image_df["label"] == class_name, "image_path"].iloc[0]
    img = Image.open(sample_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(class_name, fontsize=9)
    ax.axis("off")
for ax in axes.ravel()[len(sample_classes):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Block 6 — Define image feature extraction (HOG + colour histogram)

In [ ]:
def extract_image_features(image_path, image_size=IMAGE_SIZE):
    """Convert an RGB leaf image into HOG and RGB colour-histogram features."""
    try:
        image = Image.open(image_path).convert("RGB").resize(image_size)
        rgb = np.asarray(image, dtype=np.uint8)

        gray = rgb2gray(rgb)
        hog_features = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            feature_vector=True
        )

        colour_features = []
        for channel in range(3):
            hist, _ = np.histogram(
                rgb[:, :, channel], bins=32, range=(0, 256), density=True
            )
            colour_features.extend(hist)

        return np.concatenate([hog_features, np.asarray(colour_features, dtype=np.float32)])
    except Exception as exc:
        print(f"Could not process {image_path}: {exc}")
        return None

print("Feature extraction function is ready.")

## Block 7 — Extract features from all selected images

In [ ]:
feature_rows = []
valid_labels = []
valid_paths = []

for row in tqdm(image_df.itertuples(index=False), total=len(image_df), desc="Extracting features"):
    features = extract_image_features(row.image_path)
    if features is not None:
        feature_rows.append(features)
        valid_labels.append(row.label)
        valid_paths.append(row.image_path)

X = np.asarray(feature_rows, dtype=np.float32)
y_text = np.asarray(valid_labels)

print("Feature matrix shape:", X.shape)
print("Labels shape:", y_text.shape)
print("Images successfully processed:", len(valid_paths))

## Block 8 — Encode target labels

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

class_names = label_encoder.classes_
print("Number of encoded classes:", len(class_names))
print("Classes:", list(class_names))

## Block 9 — Split into training, validation and test sets

In [ ]:
# 70% training, 15% validation, 15% testing.
# Stratification keeps class proportions similar in each split.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print("Training:", X_train.shape, "Validation:", X_val.shape, "Testing:", X_test.shape)
print("Split percentages:",
      round(len(X_train)/len(X)*100, 1),
      round(len(X_val)/len(X)*100, 1),
      round(len(X_test)/len(X)*100, 1))

## Block 10 — Define 12 machine learning algorithms

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Linear SVM": LinearSVC(random_state=RANDOM_STATE, max_iter=5000),
    "RBF SVM": SVC(kernel="rbf", C=3, gamma="scale"),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Gaussian Naive Bayes": GaussianNB(),
    "SGD Classifier": SGDClassifier(random_state=RANDOM_STATE, max_iter=2000, tol=1e-3),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(max_iter=100, random_state=RANDOM_STATE)
}

print("Number of models:", len(models))
for name in models:
    print("-", name)

## Block 11 — Train models and compare validation performance

In [ ]:
validation_results = []
trained_models = {}

for name, estimator in models.items():
    print(f"Training: {name}")
    # Scaling is fitted on training data only; no validation/test leakage.
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", clone(estimator))
    ])

    start_time = time.time()
    pipeline.fit(X_train, y_train)
    training_seconds = time.time() - start_time

    val_pred = pipeline.predict(X_val)
    validation_results.append({
        "Model": name,
        "Validation Accuracy": accuracy_score(y_val, val_pred),
        "Validation Precision (weighted)": precision_score(y_val, val_pred, average="weighted", zero_division=0),
        "Validation Recall (weighted)": recall_score(y_val, val_pred, average="weighted", zero_division=0),
        "Validation F1 (weighted)": f1_score(y_val, val_pred, average="weighted", zero_division=0),
        "Training Time (sec)": training_seconds
    })
    trained_models[name] = pipeline

validation_df = pd.DataFrame(validation_results).sort_values(
    "Validation F1 (weighted)", ascending=False
).reset_index(drop=True)

display(validation_df.style.format({
    "Validation Accuracy": "{:.4f}",
    "Validation Precision (weighted)": "{:.4f}",
    "Validation Recall (weighted)": "{:.4f}",
    "Validation F1 (weighted)": "{:.4f}",
    "Training Time (sec)": "{:.2f}"
}))

## Block 12 — Plot validation model comparison

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=validation_df, x="Validation F1 (weighted)", y="Model")
plt.title("Validation F1-score Comparison — 12 ML Models")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## Block 13 — Select the best model using validation F1-score

In [ ]:
best_model_name = validation_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best model selected using validation data:", best_model_name)
print("Validation F1-score:",
      round(validation_df.iloc[0]["Validation F1 (weighted)"], 4))

## Block 14 — Refit the selected model on training + validation data

In [ ]:
# The test set remains untouched until final evaluation.
X_development = np.concatenate([X_train, X_val], axis=0)
y_development = np.concatenate([y_train, y_val], axis=0)

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", clone(models[best_model_name]))
])
final_model.fit(X_development, y_development)
print("Selected model retrained on combined training + validation data.")

## Block 15 — Final evaluation on the test set

In [ ]:
test_pred = final_model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred, average="weighted", zero_division=0)
test_recall = recall_score(y_test, test_pred, average="weighted", zero_division=0)
test_f1 = f1_score(y_test, test_pred, average="weighted", zero_division=0)

final_test_results = pd.DataFrame([{
    "Best Model": best_model_name,
    "Test Accuracy": test_accuracy,
    "Test Precision (weighted)": test_precision,
    "Test Recall (weighted)": test_recall,
    "Test F1-score (weighted)": test_f1
}])
display(final_test_results.style.format({
    "Test Accuracy": "{:.4f}",
    "Test Precision (weighted)": "{:.4f}",
    "Test Recall (weighted)": "{:.4f}",
    "Test F1-score (weighted)": "{:.4f}"
}))

## Block 16 — Classification report

In [ ]:
print(classification_report(
    y_test, test_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0
))

## Block 17 — Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, test_pred, labels=np.arange(len(class_names)))
plt.figure(figsize=(max(12, len(class_names)*0.45), max(10, len(class_names)*0.4)))
sns.heatmap(cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.show()

## Block 18 — Compare validation metrics in one chart

In [ ]:
metric_cols = [
    "Validation Accuracy", "Validation Precision (weighted)",
    "Validation Recall (weighted)", "Validation F1 (weighted)"
]
plot_df = validation_df.melt(
    id_vars="Model", value_vars=metric_cols,
    var_name="Metric", value_name="Score"
)
plt.figure(figsize=(15, 7))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric")
plt.title("Validation Metrics — Model Comparison")
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Block 19 — Save models, encoders and result files

In [ ]:
OUTPUT_DIR = Path("plantvillage_ml_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(final_model, OUTPUT_DIR / "best_plant_disease_model.joblib")
joblib.dump(label_encoder, OUTPUT_DIR / "label_encoder.joblib")

validation_df.to_csv(OUTPUT_DIR / "validation_model_comparison.csv", index=False)
final_test_results.to_csv(OUTPUT_DIR / "final_test_results.csv", index=False)

with open(OUTPUT_DIR / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(list(class_names), f, indent=2)

print("Saved output files in:", OUTPUT_DIR.resolve())
print("Files:", [p.name for p in OUTPUT_DIR.iterdir()])

## Block 20 — Predict the disease class for a new leaf image

In [ ]:
def predict_leaf_disease(image_path):
    features = extract_image_features(image_path).reshape(1, -1)
    predicted_encoded = final_model.predict(features)[0]
    predicted_class = label_encoder.inverse_transform([predicted_encoded])[0]

    # Show confidence only if the classifier supports predict_proba.
    classifier = final_model.named_steps["classifier"]
    confidence = None
    if hasattr(final_model, "predict_proba"):
        confidence = float(np.max(final_model.predict_proba(features)))

    return predicted_class, confidence

# Replace with the path to a new leaf image.
NEW_IMAGE_PATH = r"D:\path\to\your\new_leaf_image.jpg"

if Path(NEW_IMAGE_PATH).exists():
    prediction, confidence = predict_leaf_disease(NEW_IMAGE_PATH)
    print("Predicted disease class:", prediction)
    print("Model probability:", confidence if confidence is not None else
          "Not available for this classifier")
else:
    print("Update NEW_IMAGE_PATH to test a new image.")

## Block 21 — Prepare a concise experiment summary
Run this after all previous blocks.

In [ ]:
summary = {
    "Dataset folder": str(DATASET_DIR),
    "Number of images processed": int(len(X)),
    "Number of classes": int(len(class_names)),
    "Number of features": int(X.shape[1]),
    "Training images": int(len(X_train)),
    "Validation images": int(len(X_val)),
    "Test images": int(len(X_test)),
    "Algorithms compared": int(len(models)),
    "Best model (validation)": best_model_name,
    "Final test accuracy": round(float(test_accuracy), 4),
    "Final test precision": round(float(test_precision), 4),
    "Final test recall": round(float(test_recall), 4),
    "Final test F1-score": round(float(test_f1), 4)
}
print(json.dumps(summary, indent=2))